Week 6 (02/04/2025) - CLAHE and automatic feature extraction

In [ ]:
import cv2
import numpy as np

# It is also possible to enhance coloured images by changing the colour space.
img = cv2.imread("/home/gianmaria/01-Data/lena.png")

# Change the colour space from BGR to HSV (hue, saturation, value)
hsv_img = cv2.cvtColor(img,cv2.COLOR_BGR2HSV)
h,s,v = cv2.split(hsv_img) # h is the colour, s is the intensity, v is the brightness
eq_v = cv2.equalizeHist(v) # only the brightness is adjusted

img_eq = cv2.merge([h,s,eq_v])
img_eq = cv2.cvtColor(img_eq, cv2.COLOR_HSV2BGR)

show_img = np.hstack([img,img_eq])
cv2.imshow("equalization",show_img)
cv2.waitKey(0)
cv2.closeAllWindows()

In [ ]:
import cv2
import numpy as np

img = cv2.imread("/home/gianmaria/01-Data/tsukuba.png", 0) # load the grayscale version of the image
eq_img = cv2.equalizeHist(img)

show_img = np.hstack([img, eq_img])
cv2.imshow("equalization",show_img)
cv2.waitKey(0)
cv2.closeAllWindows()

# N.B.: This code leads to a loss of detail in correspondence of the statue.

The main issue of image equalization lies in the fact that ot typically results in the loss of details.
Therefore, by considering the image as a group of small square tiles, it is possible to perform equalization on each tile in order to reduce the loss of details.

A first improvement of image equalization is given by Contrast Limited Adaptive Histogram Equalization (CLAHE).

In [ ]:
import cv2
import numpy as np

img = cv2.imread("/home/gianmaria/01-Data/tsukuba.png", 0) # load the grayscale version of the image

# Start by instantiating the CLAHE algorithm.
clahe = cv2.createCLAHE(clipLimit = 2.0, tileGridSize = (8, 8)) # create a CLAHE with clip limit 2 that uses 8x8 tiles
clahe_img = clahe.apply(img) # apply the CLAHE instance to the input image
eq_img = cv2.equalizeHist(img)

show_img = np.hstack([img, eq_img, clahe_img])
cv2.imshow("equalization",show_img)
cv2.waitKey(0)
cv2.closeAllWindows()

Generally speaking, a feature is a characteristic that is strongly descriptive for a problem.
In the case of images, a feature is a point that has a particular meaning.

Automatic feature extraction consists in finding a way to get these features from an image.
These features are contained in the keypoint vector, which contains information about image points (such as coordinates), and in the descriptor vector, which describes the features of the found points.

Depending on the descriptor vector and on distance computation for vector comparison, feature extractor can be binary (the descriptor vector is a binary stream and comparison employs the Hamming distance) or non-binary (the descriptor vector is a stream of floats and comparison employs the Euclidean distance).

SIFT/SURF Feature Extractors (non-binary)

Generally speaking, the "Gaussian Pyramid" operation consists in applying the Gaussian blur (or another operation) various times in order to "layer" the image.
Therefore, the more layers a feature appears in, the stronger it is.

The SIFT algorithm extracts robust points that will stay in the picture even after many transformations on the whole image.
Generally speaking, these points are found in correspondence of details, whereas no feature is typically extracted from the background.

Overall, when using this algorithm, the stronger the feature, the larger the circle representing it.

N.B.: SURF is a faster, patented version of SIFT extraction.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img = cv2.imread("/home/gianmaria/01-Data/lena.png")

f_ex = cv2.SIFT_create()

# Now, compute the descriptor and keypoint arrays.
keypoints, descriptor = f_ex.detectAndCompute(img, None) # it is possible to specify a mask to extract keypoints only in correspondence of the mask.

# Plot the keypoints.
cv2.drawKeypoints(img, keypoints, img, (255, 0, 0), cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS) # draw blue keypoints and recover all their information

# Show the image and keypoints.
cv2.imshow("Keypoints", img)
cv2.waitKey(0)
cv2.closeAllWindows()

# N.B.: Trying to use SURF results in an error because it is patented.
#       However, it is possible to access it using the extra package and setting OPENCV_ENABLE_NONFREE for manual compiling.

ORB Feature Extractor (binary)

Generally speaking, binary descriptors are faster to compute but less robust compared to non-binary descriptors.

Most particularly, the ORB extractor tends to focus on a given number of points to extract and then loops over them, making the algorithm redundant as the same points will be extracted more times.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img = cv2.imread("/home/gianmaria/01-Data/lena.png")

f_ex = cv2.ORB_create(200) # just take 200 points (this helps to reduce redunancy)

# Now, compute the descriptor and keypoint arrays.
keypoints, descriptor = f_ex.detectAndCompute(img, None) # it is possible to specify a mask to extract keypoints only in correspondence of the mask.

# Plot the keypoints.
cv2.drawKeypoints(img, keypoints, img, (255, 0, 0), cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS) # draw blue keypoints and recover all their information

# Show the image and keypoints.
cv2.imshow("Keypoints", img)
cv2.waitKey(0)
cv2.closeAllWindows()

AKAZE Feature Extractor (binary)

This algorithm is similar to ORB extraction but uses different operations, yielding a different a extraction as AKAZE extraction tends to be more sparse compared to ORB extraction.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img = cv2.imread("/home/gianmaria/01-Data/lena.png")

f_ex = cv2.AKAZE_create() # just take 50 points

# Now, compute the descriptor and keypoint arrays.
keypoints, descriptor = f_ex.detectAndCompute(img, None) # it is possible to specify a mask to extract keypoints only in correspondence of the mask.

# Plot the keypoints.
cv2.drawKeypoints(img, keypoints, img, (255, 0, 0), cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS) # draw blue keypoints and recover all their information

# Show the image and keypoints.
cv2.imshow("Keypoints", img)
cv2.waitKey(0)
cv2.closeAllWindows()

Observe that it is also possible to "mix and match" different extraction algorithms by deriving the keypoint and descriptor vectors in two different moments through a dedicated matcher algorithm.